# LayerNorm Mechanics    How LayerNorm shapes feature dynamics — scaling, suppression, directionality bias, and whitening.LayerNorm is often treated as a mere normalization step, but its learned weights (w, b) actively shape which features survive and how they're weighted in downstream computation.

## Why This Matters

LayerNorm is not just a normalization step — it actively shapes which features get amplified and which get suppressed. Understanding LayerNorm's feature scaling, directionality bias, and gradient flow effects is essential because every component's output passes through it.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformerfrom irtk.norm_mechanics import (    feature_scaling_by_norm,    norm_directionality_bias,    pre_vs_post_norm_effect,    norm_gradient_flow,    feature_whitening_by_layer,)cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])

## Feature Scaling by NormHow does LayerNorm change the variance of each feature dimension? Dimensions with high pre-norm variance get compressed, while low-variance dimensions get amplified.

In [ ]:
result = feature_scaling_by_norm(model, tokens, layer=0)print(f"Mean scaling: {result['mean_scaling']:.4f}")print(f"Most amplified dim: {result['most_amplified']}")print(f"Most suppressed dim: {result['most_suppressed']}")print(f"Scaling range: [{result['scaling_factors'].min():.4f}, {result['scaling_factors'].max():.4f}]")

## Directionality BiasLayerNorm weights create an implicit directional preference — some features are amplified more than others by the learned affine transform.

In [ ]:
bias = norm_directionality_bias(model, tokens, layer=0)print(f"Anisotropy (max/min |w|): {bias['anisotropy']:.2f}")print(f"Weight range: {bias['weight_magnitude_range']}")print(f"Weight direction norm: {np.linalg.norm(bias['weight_direction']):.4f}")print(f"Bias norm: {np.linalg.norm(bias['bias_direction']):.4f}")

## Pre vs Post Norm EffectHow much does LayerNorm actually change the representation? High cosine similarity means normalization preserves direction; large rank changes indicate structural reorganization.

In [ ]:
effect = pre_vs_post_norm_effect(model, tokens, layer=0)print(f"Cosine similarity (pre vs post): {effect['cosine_similarity']:.4f}")print(f"Norm ratio (post/pre): {effect['norm_ratio']:.4f}")print(f"Effective rank before: {effect['rank_pre']:.2f}")print(f"Effective rank after: {effect['rank_post']:.2f}")

## Gradient Flow Through NormLayerNorm can amplify or attenuate gradients for different feature dimensions, affecting which dimensions get updated most during training.

In [ ]:
flow = norm_gradient_flow(model, tokens, layer=0)print(f"Mean gradient amplification: {flow['mean_amplification']:.4f}")print(f"Pre-norm grad range: [{flow['pre_norm_grad_norms'].min():.6f}, {flow['pre_norm_grad_norms'].max():.6f}]")print(f"Post-norm grad range: [{flow['post_norm_grad_norms'].min():.6f}, {flow['post_norm_grad_norms'].max():.6f}]")

## Feature Whitening by LayerDoes LayerNorm decorrelate features? Compare off-diagonal correlations before and after normalization at each layer.

In [ ]:
whitening = feature_whitening_by_layer(model, tokens)for l in range(len(whitening['pre_correlations'])):    pre = whitening['pre_correlations'][l]    post = whitening['post_correlations'][l]    effect = whitening['whitening_effect'][l]    print(f"Layer {l}: pre={pre:.4f} -> post={post:.4f}, whitening={effect:.4f}")print(f"Most whitened layer: {whitening['most_whitened_layer']}")